In [ ]:
# NEED to "conda activate vapor" before lauching the jupyter lab at the server side
# https://projectpythia.org/vapor-python-cookbook/README.html
# https://projectpythia.org/vapor-python-cookbook/notebooks/keyframing_example.html

from vapor import session, dataset, renderer
from vapor.animation import Animation


In [ ]:
import os
import requests
import zipfile
url = 'https://data.rda.ucar.edu/ds897.7/Katrina.zip'
extract_to = './data'
zip_name = "Katrina.zip"
data_file = './data/wrfout_d02_2005-08-29_02.nc'

# Check if the data file already exists
if not os.path.exists(data_file):
    # Download zip
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(zip_name, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
    # Extract the file
    with zipfile.ZipFile(zip_name, 'r') as zip_ref:
        zip_ref.extractall(extract_to)

    # Clean up the zip file
    os.remove(zip_name)

    print(f"Data downloaded and extracted to {data_file}")
else:
    print(f"Data file already exists at {data_file}, skipping download and extraction.")

In [ ]:
ses = session.Session()
data = ses.OpenDataset(dataset.WRF, ["data/wrfout_d02_2005-08-29_02.nc"])
land = data.NewRenderer(renderer.ImageRenderer) # Render land image
land.GetTransform().SetTranslations([0,0,100])
land.SetHeightVariableName("HGT")
wind = data.NewRenderer(renderer.TwoDDataRenderer) # Render U10
wind.SetVariableName("U10")
clouds = data.NewRenderer(renderer.VolumeRenderer) # Render clouds
clouds.SetVariableName("QCLOUD")
clouds_tf = clouds.GetTransferFunction("QCLOUD")
clouds_tf.LoadBuiltinColormap("Sequential/BlackWhite")
clouds_tf.SetColorRGBList([(r, g, b) for r, g, b, _ in 
                           list(reversed(clouds_tf.GetMatPlotLibColormap().colors))])
clouds_tf.SetOpacityControlPoints([[0,0],[0.00001,0.01], [0.0001, 0.1], [0.0010,0.9]])

In [ ]:
# The camera position of each keyframe
positions = [
    [-1190444.44426004, 1882360.85954653, 770176.40842364], # Keyframe 1
    [-1172384.15238047, 2813172.26639064, 355291.41877028], # Keyframe 2
    [-968784.32993129, 3056725.58106798, -34317.16158186], # ...
    [-733144.08018801, 2929790.21696698, -32984.22588893],
    [-691781.20449513, 2442083.68616993, -47289.68751812]
]

# The camera target for each keyframe
targets = [
    [-420811.28125, 2737271.75, 5699.78515597], # Keyframe 1
    [-420811.28125, 2737271.75, 15699.78515597], # ...
    [-420811.28125, 2737271.75, 15699.78515597],
    [-420811.28125, 2737271.75, 15699.78515597],
    [-420811.28125, 2737271.75, 15699.78515597]
]

# The up vector for each keyframe
ups = [
    [0.41853764, 0.35630071, 0.83538976],
    [ 0.39861183, -0.08972356, 0.91272027],
    [-0.08058301, 0.02890014, 0.99632884],
    [-0.080583, 0.02890014, 0.99632884],
    [-0.0964622, -0.0632074, 0.99332767]
]

In [ ]:
os.makedirs("keyframes", exist_ok=True) # Make directory for keyframes if it doesn't exist
for i, position, target, up in zip(range(1, len(positions)+1), positions, targets, ups):
    ses.GetCamera().LookAt(position, target, up)
    ses.Save(f"./keyframes/keyframe{i}.vs3")

In [ ]:
sessions = [
    "keyframes/keyframe1.vs3",
    "keyframes/keyframe2.vs3",
    "keyframes/keyframe3.vs3",
    "keyframes/keyframe4.vs3",
    "keyframes/keyframe5.vs3",
]
steps = [40,30,30,30]

In [ ]:
from vapor.utils import keyframing
anim = keyframing.animate_camera_keyframes(sessions, steps)
anim.Show()